In [1]:
import pandas as pd
import numpy as np
import h3
import matplotlib.pyplot as plt
import seaborn as sns
import holidays
from pathlib import Path

from sklearn.svm import SVR, LinearSVR
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
print("Loading final grid dataset...")
taxi_agg = pd.read_parquet("../data/taxi_agg.parquet")
print(f"Final grid loaded. Shape: {taxi_agg.shape}")
taxi_agg.head()

Loading final grid dataset...
Final grid loaded. Shape: (9272950, 12)


,hour,h3_index,Total_Trip_Start,Unique Taxis,AvgTripSeconds,AvgTripMiles,AvgFare,MostCommonCompany,CompanyCount,PickupLatitude,PickupLongitude,Total_Trip_End
0,2024-01-01,882664cad7fffff,2.0,2.0,425.3300,2.400000,1812.500000,City Service,2.0,41.885281,-87.657233,1.0
1,2024-01-01,88275934edfffff,7.0,7.0,1.7400,17.370000,5821.428571,Flash Cab,6.0,41.979071,-87.903040,0.0
2,2024-01-01,882664c1e7fffff,8.0,8.0,440.6800,0.587500,1168.500000,Chicago Independents,6.0,41.892042,-87.631864,5.0
3,2024-01-01,882664c1adfffff,1.0,1.0,2.8790,3.290000,2175.000000,Flash Cab,1.0,41.885300,-87.642808,0.0
4,2024-01-01,882664c1e1fffff,15.0,15.0,294.7794,1.155333,1502.666667,Chicago Independents,7.0,41.893336,-87.626569,7.0


In [3]:
# Calculate distance to Chicago Loop (downtown center)
def haversine_distance(lat1, lon1, lat2=41.8781, lon2=-87.6298):
    r = 6371 # earth radius in km
    phi1 = np.radians(lat1)
    phi2 = np.radians(lat2)
    delta_phi = np.radians(lat2 - lat1)
    delta_lambda = np.radians(lon2 - lon1)
    a = np.sin(delta_phi / 2)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(delta_lambda / 2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return r * c

In [4]:
mask = taxi_agg['PickupLatitude'].isnull() | taxi_agg['PickupLongitude'].isnull()
if mask.any():
    centroids = {cell: h3.cell_to_latlng(cell) for cell in taxi_agg.loc[mask, 'h3_index'].unique()}
    lat_map = {cell: latlng[0] for cell, latlng in centroids.items()}
    lon_map = {cell: latlng[1] for cell, latlng in centroids.items()}
    taxi_agg.loc[mask, 'PickupLatitude'] = taxi_agg.loc[mask, 'h3_index'].map(lat_map)
    taxi_agg.loc[mask, 'PickupLongitude'] = taxi_agg.loc[mask, 'h3_index'].map(lon_map)

taxi_agg['distance_to_loop'] = haversine_distance(taxi_agg['PickupLatitude'], taxi_agg['PickupLongitude'])
print(taxi_agg[['h3_index', 'PickupLatitude', 'PickupLongitude', 'distance_to_loop']].head(2))

          h3_index  PickupLatitude  PickupLongitude  distance_to_loop
0  882664cad7fffff       41.885281       -87.657233          2.407414
1  88275934edfffff       41.979071       -87.903040         25.238939


In [5]:
# Calendar features
taxi_agg['month'] = taxi_agg['hour'].dt.month
taxi_agg['day_of_week'] = taxi_agg['hour'].dt.weekday
taxi_agg['hour_of_day'] = taxi_agg['hour'].dt.hour
taxi_agg['is_weekend'] = (taxi_agg['day_of_week'] >= 5).astype(int)

# Cyclic temporal features
taxi_agg['hour_sin'] = np.sin(2 * np.pi * taxi_agg['hour_of_day'] / 24)
taxi_agg['hour_cos'] = np.cos(2 * np.pi * taxi_agg['hour_of_day'] / 24)
taxi_agg['month_sin'] = np.sin(2 * np.pi * taxi_agg['month'] / 12)
taxi_agg['month_cos'] = np.cos(2 * np.pi * taxi_agg['month'] / 12)

# Holidays feature
us_holidays = holidays.USA(years=taxi_agg['hour'].dt.year.unique(), state='IL')

taxi_agg['is_holiday'] = taxi_agg['hour'].dt.date.isin(us_holidays).astype(int)

In [6]:
pois_cat_wide = pd.read_csv("../data/chicago_pois_category_wide.csv")

In [7]:
pois_cat_wide.head()

,h3_index,poi_cat_automotive,poi_cat_civic_community,poi_cat_education,poi_cat_entertainment,poi_cat_finance,poi_cat_food_drink,poi_cat_grocery,poi_cat_health,poi_cat_leisure_sports,poi_cat_lodging,poi_cat_nightlife,poi_cat_services,poi_cat_shopping,poi_cat_transport
0,8826641917fffff,0,3,2,0,0,0,0,0,0,0,2,0,0,1
1,882664191dfffff,0,1,0,0,0,0,0,0,0,0,0,0,0,0
2,8826641927fffff,0,1,0,1,0,0,0,0,0,0,0,0,0,0
3,8826641931fffff,0,1,0,0,0,0,0,0,0,0,0,0,0,0
4,8826641939fffff,0,0,1,0,0,0,0,0,0,0,0,0,0,1


In [8]:

taxi_agg = taxi_agg.merge(
    pois_cat_wide, 
    left_on='h3_index',
    right_on='h3_index',
    how='left'
)

In [9]:
# TODO: Add more features for example: distance to nearest airport, public transit stops, etc.
# TODO: Add Weather features (e.g., temperature, precipitation) from the weather dataset, merged on hour and location.

In [14]:
# List all null values in the dataset
null_counts = taxi_agg.isnull().sum()
print("Null values in each column:")
print(null_counts[null_counts > 0])

taxi_agg['poi_cat_automotive'] = taxi_agg['poi_cat_automotive'].fillna(0).astype(int)
taxi_agg['poi_cat_entertainment'] = taxi_agg['poi_cat_entertainment'].fillna(0).astype(int)
taxi_agg['poi_cat_finance'] = taxi_agg['poi_cat_finance'].fillna(0).astype(int)
taxi_agg['poi_cat_food_drink'] = taxi_agg['poi_cat_food_drink'].fillna(0).astype(int)
taxi_agg['poi_cat_grocery'] = taxi_agg['poi_cat_grocery'].fillna(0).astype(int)
taxi_agg['poi_cat_health'] = taxi_agg['poi_cat_health'].fillna(0).astype(int)
taxi_agg['poi_cat_leisure_sports'] = taxi_agg['poi_cat_leisure_sports'].fillna(0).astype(int)
taxi_agg['poi_cat_lodging'] = taxi_agg['poi_cat_lodging'].fillna(0).astype(int)
taxi_agg['poi_cat_nightlife'] = taxi_agg['poi_cat_nightlife'].fillna(0).astype(int)
taxi_agg['poi_cat_services'] = taxi_agg['poi_cat_services'].fillna(0).astype(int)
taxi_agg['poi_cat_shopping'] = taxi_agg['poi_cat_shopping'].fillna(0).astype(int)
taxi_agg['poi_cat_transport'] = taxi_agg['poi_cat_transport'].fillna(0).astype(int)
taxi_agg['poi_cat_civic_community'] = taxi_agg['poi_cat_civic_community'].fillna(0).astype(int)
taxi_agg['poi_cat_education'] = taxi_agg['poi_cat_education'].fillna(0).astype(int)

null_counts = taxi_agg.isnull().sum()
print("Null values in each column:")
print(null_counts[null_counts > 0])

Null values in each column:
poi_cat_civic_community    224675
poi_cat_education          224675
dtype: int64
Null values in each column:
Series([], dtype: int64)


In [15]:
# Export aggregated feature dataset for modeling
taxi_agg.to_parquet("../data/chicago_taxi_features.parquet")